# PEEC Impedance Extraction with ngsbem

This notebook demonstrates the **Partial Element Equivalent Circuit (PEEC)** method
for computing the port impedance $Z(f)$ of a conductor, using **ngsbem** (NGSolve BEM)
for the Galerkin boundary element assembly.

## Physical problem

We consider a thin, flat conductor (e.g., a PCB trace or a bus bar) carrying
alternating current. The goal is to compute the frequency-dependent impedance
$Z(f) = R(f) + j\omega L(f)$, which captures:

- **DC resistance** from the conductor's sheet resistance $R_\square = 1/(\sigma t)$
- **Inductive reactance** from the magnetic vector potential (self and mutual inductance)
- **Capacitive effects** at high frequencies (self-resonance)

## PEEC formulation with Loop-Star decomposition

The PEEC method discretizes the surface current $\mathbf{J}$ on the conductor surface
into a set of basis functions. We use the **Loop-Star decomposition**, which separates
the current into:

- **Loop (solenoidal) basis** — divergence-free edge currents (Raviart-Thomas / RWG functions)
- **Star (irrotational) basis** — curl-free charge-producing currents (piecewise constant on cells)

### Block system

The Loop-Star PEEC system at angular frequency $\omega = 2\pi f$ reads:

$$
\begin{pmatrix}
R + j\omega L & M_{LS}^T \\
M_{LS} & P / (j\omega)
\end{pmatrix}
\begin{pmatrix}
I_\text{loop} \\
Q_\text{star}
\end{pmatrix}
=
\begin{pmatrix}
V_\text{port} \\
0
\end{pmatrix}
$$

where:
- $L_{ij} = \mu_0 \int\!\int \frac{\mathbf{J}_i(\mathbf{r}) \cdot \mathbf{J}_j(\mathbf{r}')}{4\pi|\mathbf{r}-\mathbf{r}'|} \, dS \, dS'$ — **inductance matrix** (via Laplace single-layer on `HDivSurface`)
- $P_{ij} = \frac{1}{\varepsilon_0} \int\!\int \frac{\phi_i(\mathbf{r}) \, \phi_j(\mathbf{r}')}{4\pi|\mathbf{r}-\mathbf{r}'|} \, dS \, dS'$ — **potential coefficient matrix** (via scalar single-layer on `SurfaceL2`)
- $M_{LS,ij} = \int \nabla\!\cdot\!\mathbf{J}_j \; \phi_i \, dS$ — **divergence coupling** (standard FEM bilinear form)
- $R = \text{diag}(R_\text{edge})$ — **resistance** from sheet resistance

### ngsbem implementation

The key advantage of using **ngsbem** is the natural mapping to NGSolve function spaces:

| PEEC quantity | NGSolve space | BEM operator |
|:---|:---|:---|
| Loop DOFs ($\mathbf{J}$) | `HDivSurface` (order $p$) | `LaplaceSL` |
| Star DOFs ($\phi$) | `SurfaceL2` (order $p$) | `SingleLayerPotentialOperator` |
| Coupling $M_{LS}$ | Product space | `div(u) * v * ds` (FEM) |

This provides Galerkin discretization (symmetric matrices by construction),
high-order elements, and FMM acceleration for large problems.

### Solver strategy

The Galerkin assembly produces a **complex symmetric** system ($A = A^T$, but $A \neq A^H$),
because $L$, $P$, and $R$ are all real symmetric, and the frequency scaling $j\omega$
preserves transposition symmetry. All linear solves use **LDL$^T$ factorization**
(Bunch-Kaufman), which is the optimal direct solver for complex symmetric matrices
— half the cost of LU.

> **Scalability note**: This notebook extracts dense matrices for clarity ($O(N^3)$).
> For large-scale problems, one should use matrix-free iterative solvers
> combined with ngsbem's built-in FMM acceleration.
> Although the system is complex symmetric ($A = A^T$), which in principle
> favors COCG/COCR solvers, in practice **GMRes is recommended** because
> COCG does not guarantee monotonic residual decrease and tends to diverge
> for BEM-discretized systems without a specialized spectral preconditioner
> (e.g., SSOR or incomplete Cholesky). GMRes guarantees residual minimization
> at the cost of higher memory ($O(k)$ vectors for $k$ iterations).

## 1. Setup and mesh generation

We create a rectangular plate conductor (10 mm × 10 mm, 35 µm copper)
representing a typical PCB trace or bus bar cross-section.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

# Import the PEEC solver module (ngbem_peec.py should be in the same directory)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__))
                if '__file__' in dir() else os.getcwd())

from ngbem_peec import NGBEMPEECSolver, create_plate_mesh, get_mesh_triangles

# Physical constants
MU_0 = 4.0 * np.pi * 1e-7   # H/m
EPS_0 = 8.854187817e-12      # F/m

In [ ]:
# Conductor geometry
width = 0.01       # 10 mm
height = 0.01      # 10 mm
maxh = 0.003       # ~3 mm element size
thickness = 35e-6  # 35 um (1 oz copper)
sigma = 5.8e7      # Copper conductivity [S/m]

# Generate surface mesh via Netgen OCC
mesh = create_plate_mesh(width, height, maxh, label="conductor")

# Extract triangles for visualization
triangles, areas = get_mesh_triangles(mesh)
n_tri = len(triangles)
total_area = np.sum(areas)

print(f"Conductor: {width*1e3:.0f} mm x {height*1e3:.0f} mm, t = {thickness*1e6:.0f} um")
print(f"Mesh: {n_tri} triangles, total area = {total_area*1e6:.2f} mm²")
print(f"Sheet resistance: R_sq = {1/(sigma*thickness)*1e3:.2f} mΩ/sq")

In [ ]:
# Visualize the mesh
from matplotlib.collections import PolyCollection

fig, ax = plt.subplots(1, 1, figsize=(6, 6))

verts_2d = [[[v[0]*1e3, v[1]*1e3] for v in tri] for tri in triangles]
pc = PolyCollection(verts_2d, edgecolors='k', facecolors='lightskyblue',
                     linewidths=0.8)
ax.add_collection(pc)
ax.set_xlim(-0.5, width*1e3 + 0.5)
ax.set_ylim(-0.5, height*1e3 + 0.5)
ax.set_aspect('equal')
ax.set_xlabel('x [mm]')
ax.set_ylabel('y [mm]')
ax.set_title(f'Conductor surface mesh ({n_tri} triangles)')
plt.tight_layout()
plt.show()

## 2. BEM matrix assembly

We assemble the PEEC matrices using ngsbem:

1. **Inductance $L$**: Laplace single-layer BEM operator on `HDivSurface` (edge DOFs)
2. **Potential coefficients $P$**: Scalar single-layer on `SurfaceL2` (cell DOFs)
3. **Loop-Star coupling $M_{LS}$**: Divergence bilinear form (FEM, not BEM)
4. **Resistance $R$**: From sheet resistance $R_\square = 1/(\sigma t)$

In [ ]:
# Create solver and assemble matrices
solver = NGBEMPEECSolver(mesh, conductor_label="conductor",
                          sigma=sigma, thickness=thickness,
                          order=0, intorder=5)
solver.assemble()

print(f"Loop DOFs (HDivSurface edges): {solver.n_loop}")
print(f"Star DOFs (SurfaceL2 cells):   {solver.n_star}")
print(f"Assembly time: {solver.t_assemble:.3f} s")

### Matrix properties

The Galerkin BEM matrices have important properties:
- $L$ is **real symmetric positive semi-definite** (from the Laplace single-layer kernel)
- $P$ is **real symmetric positive definite** (from the scalar single-layer potential)
- $M_{LS}$ is **real sparse** (each edge connects at most 2 triangles)

At each frequency, the assembled system $R + j\omega L$ is **complex symmetric** ($A = A^T$),
since both $R$ and $L$ are real symmetric. This structure is preserved through
the Schur complement, enabling efficient LDL$^T$ (Bunch-Kaufman) factorization.

In [ ]:
# Verify matrix properties
L_eig = np.linalg.eigvalsh(solver.L)
P_eig = np.linalg.eigvalsh(solver.P)
M_nnz = np.count_nonzero(np.abs(solver.M_LS) > 1e-15)

print("L (inductance):")
print(f"  Symmetry error ||L-L^T||/||L|| = {np.linalg.norm(solver.L - solver.L.T)/np.linalg.norm(solver.L):.2e}")
print(f"  Eigenvalue range: [{L_eig[0]:.4e}, {L_eig[-1]:.4e}] H")
print(f"  All eigenvalues >= 0: {np.all(L_eig >= -1e-15*np.max(np.abs(L_eig)))}")
print()
print("P (potential coefficients):")
print(f"  Symmetry error ||P-P^T||/||P|| = {np.linalg.norm(solver.P - solver.P.T)/np.linalg.norm(solver.P):.2e}")
print(f"  Eigenvalue range: [{P_eig[0]:.4e}, {P_eig[-1]:.4e}] 1/F")
print(f"  All eigenvalues > 0: {np.all(P_eig > 0)}")
print()
print(f"M_LS (coupling): {M_nnz} nonzeros / {solver.M_LS.size} entries")
print(f"  Sparsity: {(1 - M_nnz/solver.M_LS.size)*100:.1f}%")

### Loop-Star visualization

The PEEC solution separates into two physically distinct quantities:

- **Loop currents** $I_\text{loop}$ — solenoidal (divergence-free) current flowing
  along mesh edges. These are the `HDivSurface` DOFs and carry the inductive energy.
- **Star charges** $Q_\text{star}$ — irrotational charge accumulation on mesh cells.
  These are the `SurfaceL2` DOFs and carry the capacitive energy.

We can visualize both on the mesh using NGSolve `GridFunction` projection.

In [ ]:
from ngbem_peec import draw_peec_model, draw_loop_star

# Show the PEEC mesh (triangles = Star DOFs, edges = Loop DOFs)
draw_peec_model(mesh, solver)

# Solve and visualize Loop-Star decomposition at 100 kHz
gf_current, gf_charge = draw_loop_star(solver, 1e5, mesh)

## 3. MQS impedance sweep

In the **magneto-quasi-static (MQS)** regime, capacitive effects are negligible.
Only the Loop DOFs contribute, and the impedance reduces to:

$$Z_\text{MQS}(\omega) = \frac{1}{\mathbf{e}^T \, (R + j\omega L)^{-1} \, \mathbf{e}}$$

where $\mathbf{e}$ is a port excitation vector that distributes the applied voltage
across all edge DOFs. For a single, simply-connected conductor plate (one port),
we use a uniform excitation $\mathbf{e} = \mathbf{1}/N_\text{edge}$, which
approximates the physical current distribution at low frequencies.

This is the primary operating regime for PEEC: power electronics (DC–1 MHz),
induction heating, and wireless power transfer applications.

In [ ]:
# MQS frequency sweep
freqs_mqs = np.logspace(1, 6, 50)  # 10 Hz to 1 MHz
Z_mqs = solver.solve_frequency(freqs_mqs, mode='mqs')

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Magnitude
axes[0].loglog(freqs_mqs, np.abs(Z_mqs), 'b-', linewidth=1.5)
axes[0].set_ylabel('|Z| [$\Omega$]')
axes[0].set_title('MQS Impedance (Loop-only)')
axes[0].grid(True, which='both', alpha=0.3)

# Phase
axes[1].semilogx(freqs_mqs, np.angle(Z_mqs, deg=True), 'r-', linewidth=1.5)
axes[1].set_xlabel('Frequency [Hz]')
axes[1].set_ylabel('Phase [deg]')
axes[1].set_ylim([-5, 95])
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# Print a few values
for f, Z in zip(freqs_mqs[::10], Z_mqs[::10]):
    L_nH = np.imag(Z) / (2*np.pi*f) * 1e9
    print(f"  f = {f:.1e} Hz:  R = {Z.real:.4e} Ohm,  L = {L_nH:.2f} nH,  |Z| = {np.abs(Z):.4e} Ohm")

## 4. Full Loop-Star impedance (with capacitive effects)

Including the Star DOFs captures the capacitive self-resonance of the conductor.
The port impedance is obtained via the **Schur complement**:

$$
Z_\text{eff} = (R + j\omega L) - M_{LS}^T \left(\frac{P}{j\omega}\right)^{-1} M_{LS}
$$

At low frequencies, $P/(j\omega) \to \infty$ and the capacitive correction vanishes
(recovering MQS). At high frequencies, the capacitive term dominates and the
impedance transitions from inductive to capacitive — this is the **self-resonance**.

We compare the full Loop-Star result against the MQS approximation.
In the low-frequency regime (up to 1 MHz), both should agree closely,
confirming that the simpler MQS model is sufficient for this application.

In [ ]:
# Full Loop-Star frequency sweep (same range as MQS for comparison)
freqs_full = np.logspace(1, 6, 50)  # 10 Hz to 1 MHz
Z_full = solver.solve_frequency(freqs_full, mode='full')

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

# Magnitude
axes[0].loglog(freqs_full, np.abs(Z_full), 'b-', linewidth=1.5, label='Full Loop-Star')
axes[0].loglog(freqs_mqs, np.abs(Z_mqs), 'g--', linewidth=1, alpha=0.7, label='MQS (Loop-only)')
axes[0].set_ylabel('|Z| [$\Omega$]')
axes[0].set_title('Full Loop-Star vs MQS Impedance')
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.3)

# Phase
axes[1].semilogx(freqs_full, np.angle(Z_full, deg=True), 'b-', linewidth=1.5, label='Full Loop-Star')
axes[1].semilogx(freqs_mqs, np.angle(Z_mqs, deg=True), 'g--', linewidth=1, alpha=0.7, label='MQS')
axes[1].set_xlabel('Frequency [Hz]')
axes[1].set_ylabel('Phase [deg]')
axes[1].set_ylim([-5, 95])
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# Quantify MQS vs Full Loop-Star agreement
rel_diff = np.abs(Z_full - Z_mqs) / np.abs(Z_mqs) * 100
print(f"MQS vs Full Loop-Star relative difference:")
print(f"  Max: {np.max(rel_diff):.4f}%")
print(f"  Mean: {np.mean(rel_diff):.4f}%")
print("=> Capacitive effects are negligible in this frequency range.")

## 5. Vector potential from solved currents

The impedance $Z(f)$ is the primary PEEC output for circuit analysis.
But for coupling to external physics (magnetic cores, shields, other conductors),
the key quantity is the **vector potential** produced by the solved surface current:

$$\mathbf{A}(\mathbf{r}) = \frac{\mu_0}{4\pi} \int_S \frac{\mathbf{J}(\mathbf{r}')}{|\mathbf{r} - \mathbf{r}'|} \, dS'$$

This integral is evaluated using the same Laplace single-layer kernel
that ngsbem used to assemble the inductance matrix $L$.
The solved current $\mathbf{J}$ lives in `HDivSurface`, so we can directly
pass it to `Integrate` for potential evaluation at arbitrary observation points.

> **Low-frequency stability**: In the MQS regime ($\kappa \to 0$), the Laplace kernel
> $1/(4\pi r)$ is exact. For finite frequencies, the Helmholtz kernel
> $e^{-j\kappa r}/(4\pi r)$ is needed, but the classical BEM formulation
> suffers from a $\kappa^{-2}$ conditioning blow-up.
> The **stabilized formulation** in ngsbem resolves this by using
> the same `HDivSurface` $\times$ `SurfaceL2` product space as our PEEC
> Loop-Star decomposition, yielding $O(1)$ condition number across all frequencies.
> This means the vector potential computation remains accurate from DC to RF.

In [ ]:
from ngbem_peec import compute_vector_potential

# Solve for current distribution at 100 kHz
f_obs = 1e5
I_loop, Q_star = solver.solve_loop_star(f_obs)

# Observation points: line above conductor center, z = 1 mm to 50 mm
cx, cy = width / 2, height / 2  # conductor center
z_obs = np.linspace(1e-3, 50e-3, 30)
obs_points = np.array([[cx, cy, zi] for zi in z_obs])

print(f"Computing A at {len(obs_points)} points above conductor center...")
A = compute_vector_potential(solver, I_loop, obs_points)

# Plot |A| vs distance
A_mag = np.array([np.linalg.norm(a) for a in A])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Linear scale
axes[0].plot(z_obs * 1e3, A_mag, 'b-o', markersize=3, linewidth=1.5)
axes[0].set_xlabel('z [mm]')
axes[0].set_ylabel('|A| [T m]')
axes[0].set_title(f'Vector potential above conductor (f = {f_obs/1e3:.0f} kHz)')
axes[0].grid(True, alpha=0.3)

# Log-log scale (check 1/r decay)
axes[1].loglog(z_obs * 1e3, A_mag, 'b-o', markersize=3, linewidth=1.5,
               label='|A| (ngbem)')
# Reference: dipole far-field ~ 1/r^2
r_ref = z_obs[10:]
A_ref = A_mag[10] * (z_obs[10] / r_ref)**2
axes[1].loglog(r_ref * 1e3, A_ref, 'r--', alpha=0.5, label='1/r^2 (dipole)')
axes[1].set_xlabel('z [mm]')
axes[1].set_ylabel('|A| [T m]')
axes[1].set_title('Far-field decay')
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# Print dominant component
print(f"\nA at z = {z_obs[0]*1e3:.0f} mm: [{A[0,0]:.4e}, {A[0,1]:.4e}, {A[0,2]:.4e}] T*m")
print(f"A at z = {z_obs[-1]*1e3:.0f} mm: [{A[-1,0]:.4e}, {A[-1,1]:.4e}, {A[-1,2]:.4e}] T*m")

## 6. High-order convergence

One advantage of the ngsbem approach is the ability to use **high-order elements**.
We compare results for polynomial order $p = 0, 1, 2$ on the same mesh.

In [ ]:
# Convergence study
f_test = 1e6  # 1 MHz
orders = [0, 1, 2]

print(f"Convergence study at f = {f_test/1e6:.0f} MHz:")
print(f"{'Order':>5s}  {'n_loop':>7s}  {'n_star':>7s}  {'t_asm [ms]':>10s}  {'L [nH]':>10s}  {'Change':>10s}")
print("-" * 60)

prev_L = None
for order in orders:
    s = NGBEMPEECSolver(mesh, conductor_label="conductor",
                         sigma=sigma, thickness=thickness,
                         order=order, intorder=5 + 2*order)
    s.assemble()
    Z = s.solve_frequency(np.array([f_test]), mode='mqs')
    L_nH = np.imag(Z[0]) / (2*np.pi*f_test) * 1e9

    change = ""
    if prev_L is not None:
        pct = abs(L_nH - prev_L) / abs(prev_L) * 100
        change = f"{pct:.1f}%"
    prev_L = L_nH

    print(f"{order:>5d}  {s.n_loop:>7d}  {s.n_star:>7d}  {s.t_assemble*1e3:>10.1f}  {L_nH:>10.3f}  {change:>10s}")

## 7. Ferrite core coupling (inductance increase)

When a ferrite core ($\mu_r \gg 1$) is placed near the conductor, the magnetic flux
is concentrated, increasing the effective inductance: $L_\text{core} > L_\text{air}$.

### Image method (half-space approximation)

For a conductor above a ferrite slab, the **image method** gives the total inductance:

$$L_\text{total} = L_\text{air} \cdot \frac{2\mu_r}{\mu_r + 1}$$

This is implemented via the coupling matrix $\Delta L = L_\text{air} / (\mu_r + 1)$:
$$L_\text{total} = L_\text{air} + \Delta L \cdot (\mu_r - 1)$$

The coupling matrix $\Delta L$ is **frequency-independent** for linear materials
(computed once, reused across all frequencies). For $\mu_r \gg 1$:
$L_\text{total} \approx 2 L_\text{air}$ (inductance approximately doubles).

**Why analytical (not Radia filament coupling)?** The BEM inductance matrix $L_\text{air}$
is assembled using Galerkin RT0 surface integrals, while a filament-based $\Delta L$
uses line integrals along edges. These incompatible bases produce wildly incorrect results
when added. The analytical image method is BEM-consistent because it **scales** $L_\text{air}$
directly, preserving the matrix structure.

For complex core geometries (nonlinear, non-planar), Radia MMM can compute $\Delta L$
via the full Biot-Savart + vector potential pipeline, but that requires a PEEC formulation
with compatible (filament-based) $L_\text{air}$ matrices.

In [ ]:
# Ferrite core coupling using analytical image method
# (BEM-consistent: scales L_air directly, no basis mismatch)
try:
    from ngbem_coupled import CoupledPEECMMM, compute_delta_L

    mu_r_core = 1000.0  # ferrite permeability

    # Compute Delta_L analytically: image method for half-space
    # Delta_L = L_air / (mu_r + 1)
    # L_total = L_air + Delta_L * (mu_r - 1) = L_air * 2*mu_r/(mu_r+1)
    Delta_L = compute_delta_L(solver.L, mu_r_core)

    # Create coupled solver with static mu_r (no eddy currents in ferrite)
    coupled = CoupledPEECMMM(solver, core_model=None, mu_r=mu_r_core)
    coupled.compute_coupling_analytically(Delta_L)
    coupled.print_summary()

    # Compare air-core vs ferrite-core impedance
    freqs_cmp = np.logspace(2, 6, 30)  # 100 Hz to 1 MHz
    Z_air = solver.solve_frequency(freqs_cmp, mode='mqs')
    Z_core = coupled.solve_frequency(freqs_cmp, mode='mqs')

    # Extract inductance
    L_air_nH = np.imag(Z_air) / (2 * np.pi * freqs_cmp) * 1e9
    L_core_nH = np.imag(Z_core) / (2 * np.pi * freqs_cmp) * 1e9

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogx(freqs_cmp, L_air_nH, 'b-', linewidth=1.5, label='Air core')
    ax.semilogx(freqs_cmp, L_core_nH, 'r-', linewidth=1.5,
                label=f'Ferrite core ($\\mu_r = {mu_r_core:.0f}$)')
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('L [nH]')
    ax.set_title('Inductance: Air core vs Ferrite core (image method)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Print comparison
    idx_1k = np.argmin(np.abs(freqs_cmp - 1e3))
    L_air_val = L_air_nH[idx_1k]
    L_core_val = L_core_nH[idx_1k]
    pct_inc = (L_core_val / L_air_val - 1) * 100
    L_ratio_theory = 2 * mu_r_core / (mu_r_core + 1)
    print(f"\nL_air  @ 1 kHz = {L_air_val:.2f} nH")
    print(f"L_core @ 1 kHz = {L_core_val:.2f} nH")
    print(f"Increase: {pct_inc:.1f}% (theory: {(L_ratio_theory-1)*100:.1f}%)")
    print(f"L_core/L_air = {L_core_val/L_air_val:.4f} (theory: {L_ratio_theory:.4f})")
    print(f"=> Ferrite core approximately doubles inductance (image method limit).")

except ImportError as e:
    print(f"Missing dependency ({e}). Skipping ferrite core coupling.")
    print("This section requires ngbem_coupled.py in the same directory.")

## 8. Conducting shield coupling (inductance decrease)

A conducting shield (aluminum plate) placed near the conductor induces **eddy currents**
that oppose the source magnetic field (**Lenz's law**). This produces two measurable effects:

- **Inductance decreases**: $L_\text{shield} < L_\text{air}$ (flux cancellation from opposing eddy currents)
- **Resistance increases**: $R_\text{shield} > R_\text{air}$ (Ohmic loss in the shield adds to total loss)

The coupling matrix $\Delta Z$ is **complex** and **frequency-dependent** (unlike $\Delta L$ for ferrite):
1. Unit current in filament $j$ $\to$ $\mathbf{A}_{\text{inc},j}(\mathbf{r})$ via Biot-Savart
2. BEM + SIBC solve for shield eddy currents $\mathbf{J}_\text{shield}$
3. $\mathbf{A}_\text{scat}$ from $\mathbf{J}_\text{shield}$ at all filament centers
4. $\Delta Z_{ij} = j\omega \, \mathbf{A}_\text{scat}(\mathbf{r}_i) \cdot \hat{\ell}_i \, |\ell_i|$

The sign follows from Lenz's law: $\text{Im}(\Delta Z) < 0$ (reduced inductance)
and $\text{Re}(\Delta Z) > 0$ (added resistance).

In [ ]:
try:
    from netgen.occ import Box, Pnt, OCCGeometry
    from ngsolve import Mesh as NGMesh, BND
    from ngbem_eddy import ShieldBEMSIBC
    from ngbem_interface import extract_edge_geometry

    # Create aluminum shield plate 5 mm above conductor
    # Plate: 12x12 mm, 0.5 mm thick aluminum (sigma = 3.7e7 S/m)
    shield_plate = Box(
        Pnt(-0.006, -0.006, 0.005),
        Pnt(0.006, 0.006, 0.0055))
    shield_plate.solids.name = "conductor"
    shield_plate.faces.name = "surface"
    geo_shield = OCCGeometry(shield_plate)
    shield_mesh = NGMesh(geo_shield.GenerateMesh(maxh=0.003))

    n_bnd = sum(1 for _ in shield_mesh.Elements(BND))
    print(f"Shield mesh: {n_bnd} boundary elements")

    # Assemble shield BEM + SIBC (half-space SIBC; delta << 0.5 mm at high freq)
    shield = ShieldBEMSIBC(shield_mesh, sigma=3.7e7)
    shield.assemble(intorder=4)
    print(f"Shield assembled: {shield._loop.n_loops} loops, "
          f"{shield._loop.n_active} active DOFs")

    # Build topology dict from conductor mesh edge geometry
    edge_geom = extract_edge_geometry(mesh)
    topo_dict = {
        'segment_centers': edge_geom['centers'],
        'segment_directions': edge_geom['directions'],
        'segment_lengths': edge_geom['lengths'],
    }

    # Compute shielded impedance at several frequencies
    freqs_shield = np.array([1e3, 10e3, 100e3])
    Z_shielded = np.zeros(len(freqs_shield), dtype=complex)

    for k, f in enumerate(freqs_shield):
        omega = 2 * np.pi * f

        # Compute Delta_Z from shield coupling (BEM solve per frequency)
        Delta_Z = shield.compute_impedance_matrix(f, topo_dict)

        # Add to air-core impedance: Z_branch = R + jw*L + Delta_Z
        Z_branch = np.diag(solver.R_loop.astype(complex)) + 1j * omega * solver.L + Delta_Z

        # Port impedance via uniform excitation
        Y = np.linalg.inv(Z_branch)
        e = np.ones(solver.n_loop) / solver.n_loop
        Z_shielded[k] = 1.0 / (e @ Y @ e)

    # Compare with air-core
    Z_air_pts = solver.solve_frequency(freqs_shield, mode='mqs')
    L_air_pts = np.imag(Z_air_pts) / (2 * np.pi * freqs_shield) * 1e9    # nH
    L_shld_pts = np.imag(Z_shielded) / (2 * np.pi * freqs_shield) * 1e9  # nH
    R_air_pts = np.real(Z_air_pts) * 1e3     # mOhm
    R_shld_pts = np.real(Z_shielded) * 1e3   # mOhm

    print(f"\n{'freq':>10s}  {'L_air(nH)':>10s}  {'L_shld(nH)':>11s}  "
          f"{'R_air(mOhm)':>12s}  {'R_shld(mOhm)':>13s}")
    print("-" * 65)
    for k, f in enumerate(freqs_shield):
        print(f"{f:10.0f}  {L_air_pts[k]:10.2f}  {L_shld_pts[k]:11.2f}  "
              f"{R_air_pts[k]:12.4f}  {R_shld_pts[k]:13.4f}")

    # Physical checks (Lenz's law)
    dL = L_shld_pts - L_air_pts
    dR = R_shld_pts - R_air_pts
    print(f"\nPhysical checks (Lenz's law):")
    print(f"  L decreases with shield: {np.all(dL < 0)}")
    print(f"  R increases with shield: {np.all(dR > 0)}")
    print(f"=> Conducting shield decreases inductance and increases resistance.")

except ImportError as e:
    print(f"Missing dependency ({e}). Skipping conducting shield section.")

## 9. Connection to the stabilized EFIE formulation (Weggler)

The `HDivSurface` $\times$ `SurfaceL2` product space used for PEEC Loop-Star
is **identical** to the one used in Weggler's stabilized EFIE formulation
([Maxwell_DtN_Stabilized.ipynb](https://github.com/Weggler/docu-ngsbem/blob/main/demos/Maxwell_DtN_Stabilized.ipynb)).

### Classical EFIE: low-frequency breakdown

The classical combined operator
$$V = V_1 - \frac{1}{\kappa^2} V_2$$
where $V_1 = \texttt{HelmholtzSL}(\texttt{HDivSurface})$ and $V_2 = \texttt{HelmholtzSL}(\nabla\cdot, \nabla\cdot)$,
has condition number growing as $O(\kappa^{-2})$ as $\kappa \to 0$.
This makes iterative solvers fail in the MQS regime.

### Stabilized formulation: $O(1)$ condition number

The stabilized block system
$$\begin{pmatrix} A_\kappa & Q_\kappa \\ Q_\kappa^T & \kappa^2 V_\kappa \end{pmatrix}$$
uses exactly the same function spaces as our PEEC:

| Stabilized EFIE | PEEC Loop-Star | ngsbem space |
|:---|:---|:---|
| $A_\kappa$ (vector SL) | Inductance $L$ | `HelmholtzSL` on `HDivSurface` |
| $V_\kappa$ (scalar SL) | Potential $P$ | `HelmholtzSL` on `SurfaceL2` |
| $Q_\kappa$ (coupling) | Divergence $M_{LS}$ | `HelmholtzSL(div, SurfaceL2)` |

The key insight: multiplying $V_\kappa$ by $\kappa^2$ (instead of dividing $V_2$ by $\kappa^2$)
keeps the system well-conditioned as $\kappa \to 0$.
In the limit $\kappa \to 0$, the Helmholtz SL reduces to the Laplace SL — exactly
what our PEEC uses — and the block system becomes a well-posed saddle-point problem.

Below we verify this numerically: we compare the inductance matrix $L$ extracted
from the Laplace kernel (our PEEC, Section 2) with the $A_\kappa$ block extracted
from the Helmholtz stabilized system at small $\kappa$.

In [ ]:
from ngsolve import HDivSurface, SurfaceL2, TaskManager, ds, div
from ngsolve.bem import HelmholtzSL, LaplaceSL
from ngbem_peec import extract_dense_matrix

# Use the same mesh from Section 1
fes_hdiv = HDivSurface(mesh, order=0, complex=True)
fes_l2 = SurfaceL2(mesh, order=0, complex=True, dual_mapping=True)
fes_prod = fes_hdiv * fes_l2
(uHDiv, uL2), (vHDiv, vL2) = fes_prod.TnT()

n_loop = fes_hdiv.ndof
n_star = fes_l2.ndof
n_total = fes_prod.ndof
intorder = 8

print(f"Product space: {n_loop} loop (HDivSurface) + {n_star} star (SurfaceL2) = {n_total} DOFs")

# --- A) PEEC Laplace kernel (reference, from Section 2) ---
print("\n--- A) PEEC: Laplace SL (kappa=0, our reference) ---")
u_h, v_h = fes_hdiv.TnT()
with TaskManager():
    L_op = LaplaceSL(
        u_h.Trace() * ds(bonus_intorder=intorder)
    ) * v_h.Trace() * ds(bonus_intorder=intorder)

L_laplace = MU_0 * np.real(extract_dense_matrix(L_op.mat, n_loop))
print(f"  L_laplace: ({n_loop}x{n_loop}), diag range [{np.min(np.diag(L_laplace)):.4e}, {np.max(np.diag(L_laplace)):.4e}]")

# --- B) Stabilized Helmholtz at small kappa ---
kappa_values = [1.0, 0.1, 0.01, 0.001]

print(f"\n--- B) Stabilized Helmholtz at decreasing kappa ---")
print(f"{'kappa':>10s}  {'||L_helm-L_lap||/||L_lap||':>28s}  {'cond(stabilized)':>18s}  {'cond(classical)':>17s}")
print("-" * 80)

for kappa in kappa_values:
    with TaskManager():
        A_k = HelmholtzSL(
            uHDiv.Trace() * ds(bonus_intorder=intorder), kappa
        ) * vHDiv.Trace() * ds(bonus_intorder=intorder)

        V_k = HelmholtzSL(
            uL2 * ds(bonus_intorder=intorder), kappa
        ) * vL2 * ds(bonus_intorder=intorder)

        Q_k = HelmholtzSL(
            div(uHDiv.Trace()) * ds(bonus_intorder=intorder), kappa
        ) * vL2 * ds(bonus_intorder=intorder)

    # Stabilized system (product space)
    lhs_stab = A_k.mat + Q_k.mat + Q_k.mat.T + kappa**2 * V_k.mat
    M_stab = extract_dense_matrix(lhs_stab, n_total)

    # Extract A_kappa block (loop-loop) -> compare with Laplace L
    A_block = M_stab[:n_loop, :n_loop]
    L_helm = MU_0 * np.real(A_block)

    # L matrix comparison
    L_diff = np.linalg.norm(L_laplace - L_helm) / np.linalg.norm(L_laplace)

    # Condition number of stabilized system
    s_stab = np.linalg.svd(M_stab, compute_uv=False)
    s_nonzero = np.abs(s_stab[np.abs(s_stab) > 1e-14 * np.abs(s_stab[0])])
    cond_stab = s_nonzero[0] / s_nonzero[-1]

    # Classical EFIE condition number for comparison
    with TaskManager():
        u_cl, v_cl = fes_hdiv.TnT()
        V1 = HelmholtzSL(
            u_cl.Trace() * ds(bonus_intorder=intorder), kappa
        ) * v_cl.Trace() * ds(bonus_intorder=intorder)
        V2 = HelmholtzSL(
            div(u_cl.Trace()) * ds(bonus_intorder=intorder), kappa
        ) * div(v_cl.Trace()) * ds(bonus_intorder=intorder)
        lhs_class = V1.mat - (1.0 / (kappa**2)) * V2.mat

    M_class = extract_dense_matrix(lhs_class, n_loop)
    s_class = np.linalg.svd(M_class, compute_uv=False)
    s_cl_nz = np.abs(s_class[np.abs(s_class) > 1e-14 * np.abs(s_class[0])])
    cond_class = s_cl_nz[0] / s_cl_nz[-1]

    print(f"{kappa:10.4f}  {L_diff:28.6e}  {cond_stab:18.3e}  {cond_class:17.3e}")

print("\nConclusion:")
print("  - Helmholtz A_kappa -> Laplace L as kappa -> 0 (L matrix converges)")
print("  - Stabilized cond. number stays bounded (O(1)) at all kappa")
print("  - Classical cond. number grows as O(kappa^{-2})")
print("  => PEEC Loop-Star and stabilized EFIE share the same product-space structure")

### Physical coupling: ferrite increases L, shield decreases L

The stabilized `HDivSurface` $\times$ `SurfaceL2` framework handles
both types of material coupling seamlessly:

| Material | Effect on $L$ | Effect on $R$ | Physics | PEEC modification |
|:---|:---|:---|:---|:---|
| **Ferrite** ($\mu_r \gg 1$) | $L \uparrow$ | (negligible) | Flux concentration | $L_\text{eff} = L_\text{air} \cdot \frac{2\mu_r}{\mu_r+1}$ |
| **Conductor** ($\sigma \gg 0$) | $L \downarrow$ | $R \uparrow$ | Lenz's law (opposing eddy currents) | $Z_\text{eff} = Z_\text{air} + \Delta Z(f)$ |

- $\Delta L$ from the ferrite is **real** and **frequency-independent** (linear material),
  computed analytically via the image method (Section 7).
- $\Delta Z$ from the shield is **complex** and **frequency-dependent** (eddy currents),
  requiring a BEM+SIBC solve per frequency (Section 8).

Below we combine the results from Sections 7 and 8 into a single plot.

In [ ]:
# Combined coupling plot: air vs ferrite vs shield
# Uses results from Sections 7 (Z_core, freqs_cmp) and 8 (Z_shielded, freqs_shield)

has_ferrite = 'Z_core' in dir() and 'freqs_cmp' in dir()
has_shield = 'Z_shielded' in dir() and 'freqs_shield' in dir()

if not has_ferrite and not has_shield:
    print("Sections 7 and/or 8 did not run (missing dependencies).")
    print("Run those sections first to see the combined coupling plot.")
else:
    # Air-core reference (always available)
    freqs_ref = np.logspace(2, 6, 50)  # 100 Hz to 1 MHz
    Z_ref = solver.solve_frequency(freqs_ref, mode='mqs')
    L_ref_nH = np.imag(Z_ref) / (2 * np.pi * freqs_ref) * 1e9
    R_ref_mOhm = np.real(Z_ref) * 1e3

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # --- Left panel: Inductance ---
    ax = axes[0]
    ax.semilogx(freqs_ref, L_ref_nH, 'k-', linewidth=2, label='Air (PEEC)')

    if has_ferrite:
        L_core_plot = np.imag(Z_core) / (2 * np.pi * freqs_cmp) * 1e9
        ax.semilogx(freqs_cmp, L_core_plot, 'r-s', linewidth=1.5, markersize=4,
                    label=r'+ Ferrite ($\mu_r = 1000$)')

    if has_shield:
        L_shld_plot = np.imag(Z_shielded) / (2 * np.pi * freqs_shield) * 1e9
        ax.semilogx(freqs_shield, L_shld_plot, 'b-^', linewidth=1.5, markersize=7,
                    label='+ Al shield')

    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('L [nH]')
    ax.set_title('Inductance: material coupling')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Add arrows showing physical direction
    L_mid = np.median(L_ref_nH)
    if has_ferrite:
        ax.annotate(r'$L \uparrow$ (flux concentration)',
                    xy=(1e4, L_mid * 1.15), fontsize=9, color='red',
                    ha='center')
    if has_shield:
        ax.annotate(r"$L \downarrow$ (Lenz's law)",
                    xy=(1e4, L_mid * 0.85), fontsize=9, color='blue',
                    ha='center')

    # --- Right panel: Resistance ---
    ax = axes[1]
    ax.semilogx(freqs_ref, R_ref_mOhm, 'k-', linewidth=2, label='Air (PEEC)')

    if has_ferrite:
        R_core_plot = np.real(Z_core) * 1e3
        ax.semilogx(freqs_cmp, R_core_plot, 'r-s', linewidth=1.5, markersize=4,
                    label=r'+ Ferrite ($\mu_r = 1000$)')

    if has_shield:
        R_shld_plot = np.real(Z_shielded) * 1e3
        ax.semilogx(freqs_shield, R_shld_plot, 'b-^', linewidth=1.5, markersize=7,
                    label='+ Al shield')

    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('R [mOhm]')
    ax.set_title('Resistance: material coupling')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary table
    print("Combined coupling summary:")
    print(f"  {'':>12s}  {'L [nH]':>10s}  {'R [mOhm]':>10s}  {'Effect':>20s}")
    print("  " + "-" * 58)

    # Pick a representative frequency for comparison
    f_rep = 1e4  # 10 kHz
    idx_ref = np.argmin(np.abs(freqs_ref - f_rep))
    L_air_val = L_ref_nH[idx_ref]
    R_air_val = R_ref_mOhm[idx_ref]
    print(f"  {'Air':>12s}  {L_air_val:10.2f}  {R_air_val:10.4f}  {'(reference)':>20s}")

    if has_ferrite:
        idx_c = np.argmin(np.abs(freqs_cmp - f_rep))
        L_c = np.imag(Z_core[idx_c]) / (2 * np.pi * freqs_cmp[idx_c]) * 1e9
        R_c = np.real(Z_core[idx_c]) * 1e3
        dL_pct = (L_c / L_air_val - 1) * 100
        print(f"  {'+ Ferrite':>12s}  {L_c:10.2f}  {R_c:10.4f}  {'L +' + f'{dL_pct:.0f}%':>20s}")

    if has_shield:
        idx_s = np.argmin(np.abs(freqs_shield - f_rep))
        L_s = np.imag(Z_shielded[idx_s]) / (2 * np.pi * freqs_shield[idx_s]) * 1e9
        R_s = np.real(Z_shielded[idx_s]) * 1e3
        dL_pct = (L_s / L_air_val - 1) * 100
        dR_pct = (R_s / R_air_val - 1) * 100
        print(f"  {'+ Shield':>12s}  {L_s:10.2f}  {R_s:10.4f}  {'L ' + f'{dL_pct:.0f}%, R +{dR_pct:.0f}%':>20s}")

    print(f"\n  @ f = {f_rep/1e3:.0f} kHz")

## Summary

This notebook demonstrated the PEEC impedance extraction workflow using ngsbem:

1. **Mesh generation**: Netgen OCC creates a surface mesh of the conductor
2. **BEM assembly**: ngsbem computes $L$ (inductance) and $P$ (potential coefficients)
   via Galerkin boundary elements on `HDivSurface` and `SurfaceL2`
3. **Loop-Star visualization**: Current distribution (edges) and charge density (cells)
   projected onto NGSolve `GridFunction` for interactive inspection
4. **MQS sweep**: Loop-only impedance $Z = R + j\omega L$ for the low-frequency regime
   (10 Hz -- 1 MHz), covering power electronics, WPT, and induction heating applications
5. **Full Loop-Star**: Includes capacitive coupling via Schur complement;
   confirms that MQS is sufficient in this frequency range
6. **Vector potential**: Computes $\mathbf{A}(\mathbf{r})$ from the solved surface current
   $\mathbf{J}$, the key output for coupling to external physics (magnetic cores, shields)
7. **High-order convergence**: Increasing FE order improves accuracy on the same mesh
8. **Ferrite core coupling**: Analytical image method computes $\Delta L$ from
   BEM-consistent scaling of $L_\text{air}$
   ($\mu_r = 1000$) $\to$ inductance approximately doubles ($L_\text{core} \approx 2 L_\text{air}$)
9. **Conducting shield coupling**: BEM + SIBC computes $\Delta Z$ from eddy currents
   in an aluminum shield $\to$ inductance decreases ($L_\text{shield} < L_\text{air}$),
   resistance increases ($R_\text{shield} > R_\text{air}$)
10. **Stabilized EFIE connection**: The `HDivSurface` $\times$ `SurfaceL2` product space
   in PEEC Loop-Star is identical to Weggler's stabilized EFIE, providing $O(1)$
   condition number from DC to RF

### Key advantages of the ngsbem approach

- **Galerkin discretization** -- symmetric matrices by construction
- **High-order elements** -- convergence with $p$-refinement, not just $h$-refinement
- **Natural Loop-Star decomposition** -- no explicit basis transformation needed;
  `HDivSurface` and `SurfaceL2` are the Loop and Star spaces
- **Built-in visualization** -- `GridFunction` projection to webgui for Loop currents and Star charges
- **Vector potential output** -- solved $\mathbf{J}$ directly yields $\mathbf{A}(\mathbf{r})$
  at arbitrary observation points via `Integrate`
- **Low-frequency stability** -- the stabilized `HDivSurface` $\times$ `SurfaceL2` formulation
  maintains $O(1)$ condition number from DC to RF
- **FMM acceleration** -- available in ngsbem for large-scale problems
- **Material coupling** -- ferrite (image method / Radia MMM) and shield (BEM+SIBC) modify PEEC impedance

### References

- A. Ruehli, "Equivalent Circuit Models for Three-Dimensional Multiconductor Systems,"
  *IEEE Trans. MTT*, 1974.
- F. Andriulli et al., "A Multiplicative Calderon Preconditioner for the Electric Field
  Integral Equation," *IEEE Trans. AP*, 2008.
- J. Ostrowski et al., "ngbem -- A BEM Library for NGSolve," 2024.
- L. Weggler, "Stabilized Maxwell DtN Formulation," ngsbem demo notebook, 2026.
  [Maxwell_DtN_Stabilized.ipynb](https://github.com/Weggler/docu-ngsbem/blob/main/demos/Maxwell_DtN_Stabilized.ipynb)